# ReAct Agent — Reasoning Loop on Top of the Agentic RAG Pipeline

The agentic RAG router from `003. Agentic Router_semantic_caching_rbac.ipynb` makes **one** decision: pick one source → run one retrieval → write one answer.
That breaks on questions that need **several lookups**, **more than one source**, or **arithmetic**.

The cells below wrap the same pipeline in a minimal **ReAct** (Reason + Act) loop:

```
Thought      → what do I still need to know?
Action       → call a tool: search_10k | search_openai_docs | search_web | calculator
Observation  → the tool's result is fed back to the model
... repeat until ...
Final Answer
```

The tools reuse the pipeline's embedder and Qdrant client, with two changes that keep the agent **grounded**:

- **Top 6 chunks per 10-K search**, since the right chunk often ranks 4th–6th for a wordy query.
- **Raw chunks, not summaries.** `search_10k` / `search_openai_docs` return the retrieved text itself, labelled `(company FYyear)`. They don't pass it through another LLM, because a summarising LLM can slip memorised figures into the Observation.
- **Filtered to one filing.** `search_10k` filters Qdrant on `metadata.company` and, if the query names a year, `metadata.fiscal_year` (a Qdrant payload filter). If that filing isn't in the collection, the tool says so and lists the years that are.

Plus a prompt rule: *only state figures that appear in an Observation.* The loop itself is ~40 lines: a prompt, a regex parser, and a `for` loop.

**Questions the single-shot RAG can't answer well:**

| # | Question type | Why one-shot RAG struggles |
|---|---|---|
| 1 | According to the 10-K: which freight company Uber bought in 2021, its price, and % of revenue | **Chained**: find *which* company, then its price, then revenue, then divide. One search on the full question doesn't return the price chunk |
| 2 | Higher-revenue company (Uber 2021 vs Lyft 2022) → its R&D as % of revenue | **Chained**: the answer to part 1 decides what to look up in part 2 |
| 3 | Lyft's long-term debt (net of current portion) at end-2022, as % of 2022 revenue | Two figures from different parts of the filing (debt note / balance sheet + income statement), and obscure enough that the model can't recall them |
| 4 | Uber price-to-sales (10-K revenue ÷ live market cap) | Needs **10-K + web**; the router picks only one source |
| 5 | Uber risk factors → which Agents SDK feature helps | Needs **10-K + OpenAI docs** |
| 6 | Uber's Q4 2021 MAPCs (10-K), then its Q2 2026 MAPCs (web), then growth | The web lookup builds on the 10-K figure, then arithmetic. The router can only pick one of the two sources |

Each cell below sends the same question to the **plain agentic RAG** (`_run_rag_pipeline`, called directly so the semantic cache doesn't interfere) and then to the **ReAct agent**, so you can compare. It also prints **the 3 chunks the one-shot retrieval actually saw**. The model has memorised a lot about Uber and Lyft, so plain RAG often *sounds* right; checking its answer against those chunks shows whether it came from the documents or from memory.

## 0. Setup

Run from this `Rag_react` folder, in a venv built from `requirements.txt`. The folder needs:
- `rag_helpers.py`: the agentic RAG pipeline (router, Qdrant retrieval, SerpApi search)
- `Agentic_RAG/qdrant_data/`: the vector database (Uber FY2021 and Lyft FY2020–2022 10-Ks, plus OpenAI Agents SDK docs)
- `.env` with `openai_api_key=...` and `serp_api_key=...`

```
python3.11 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

In [1]:
import os, sys, nest_asyncio
from dotenv import load_dotenv

sys.path.insert(0, os.getcwd())  # rag_helpers.py sits next to this notebook
nest_asyncio.apply()             # lets asyncio.run() work inside Jupyter
load_dotenv(os.path.join(os.getcwd(), ".env"))

from rag_helpers import init_rag

init_rag(
    openai_api_key=os.getenv("openai_api_key") or os.getenv("OPENAI_API_KEY"),
    serp_api_key=os.getenv("serp_api_key") or os.getenv("SERP_API_KEY"),
    qdrant_path=os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data"),
)

Loading Nomic text model for Qdrant retrieval embeddings...


<All keys matched successfully>


✅ RAG pipeline ready.


## 1. Tools

Each tool takes a string and returns a string. They reuse the pipeline's embedder, Qdrant client and SerpApi search from `rag_helpers.py`:

| Tool | What it does |
|---|---|
| `search_10k` | Raw 10-K chunks, filtered to **one company** and (if named) **one fiscal year** |
| `search_openai_docs` | Raw chunks from the OpenAI Agents SDK docs |
| `search_web` | Live Google results via SerpApi |
| `calculator` | Arithmetic only (digits and `+ - * / ( ) %`) |

In [2]:
import asyncio
import inspect
import re

from qdrant_client import models

import rag_helpers

MODEL = "gpt-5.6-luna"  # same model the notebook uses
MAX_STEPS = 10


# ── Tools ─────────────────────────────────────────────────────────────────────
# Each tool takes a string and returns a string. They reuse the existing RAG pipeline's
# embedder and Qdrant client, but return the RAW retrieved chunks rather than an LLM
# summary of them: a summarising LLM can slip memorised figures into the Observation.

COMPANIES = {"uber": "Uber Technologies, Inc.", "lyft": "Lyft, Inc."}
# Fiscal years labelled in the 10k_data collection (metadata.fiscal_year).
TEN_K_YEARS = {"Uber Technologies, Inc.": [2021], "Lyft, Inc.": [2020, 2021, 2022]}


def _run(result):
    """rag_helpers mixes sync and async functions; handle both."""
    return asyncio.run(result) if inspect.iscoroutine(result) else result


def _search_chunks(collection: str, query: str, conditions=None, limit: int = 3) -> str:
    hits = _run(rag_helpers._qdrant.query_points(
        collection_name=collection,
        query=rag_helpers._get_text_embeddings(query),
        query_filter=models.Filter(must=conditions) if conditions else None,
        limit=limit,
    ))
    if not hits.points:
        return "No matching chunks."
    chunks = []
    for i, p in enumerate(hits.points, 1):
        meta = p.payload.get("metadata", {})
        label = f"{meta['company']} FY{meta.get('fiscal_year') or '?'}" if "company" in meta else "OpenAI docs"
        chunks.append(f"[{i}] ({label}) {' '.join(p.payload['content'].split())}")
    return "\n\n".join(chunks)


def search_10k(query: str) -> str:
    """Filtered to ONE company, and to the latest fiscal year named in the query (if any)."""
    companies = [name for key, name in COMPANIES.items() if key in query.lower()]
    if len(companies) != 1:
        return "Error: name exactly ONE company (Uber or Lyft) in each search."
    company = companies[0]
    conditions = [models.FieldCondition(key="metadata.company", match=models.MatchValue(value=company))]

    years = [int(y) for y in re.findall(r"\b(20\d\d)\b", query)]
    if years:
        year = max(years)  # a 10-K also reports the prior years, so use the latest one named
        if year not in TEN_K_YEARS[company]:
            return (f"There is no {company} 10-K for fiscal year {year} in the collection. "
                    f"Available fiscal years: {TEN_K_YEARS[company]}.")
        conditions.append(models.FieldCondition(key="metadata.fiscal_year", match=models.MatchValue(value=year)))

    return _search_chunks("10k_data", query, conditions, limit=6)  # top 3 often cut off the right chunk


def search_openai_docs(query: str) -> str:
    return _search_chunks("opnai_data", query)


def search_web(query: str) -> str:
    return str(_run(rag_helpers.get_internet_content(query)))


def calculator(expression: str) -> str:
    if not re.fullmatch(r"[\d\s.+\-*/()%]+", expression):
        return "Error: only numbers and + - * / ( ) % are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


TOOLS = {
    "search_10k": (search_10k, "Search the 10-K filings (Uber FY2021; Lyft FY2020-FY2022). Returns raw excerpts. "
                               "Search for ONE fact per call with a SHORT query: ONE company, the fiscal year and a few "
                               "key words, e.g. 'Lyft 2022 total revenue'."),
    "search_openai_docs": (search_openai_docs, "Search the OpenAI Agents SDK documentation."),
    "search_web": (search_web, "Search the live internet for current or external information."),
    "calculator": (calculator, "Evaluate an arithmetic expression, e.g. (17455 - 11139) / 11139 * 100"),
}

In [3]:
# The 10-K tool is filtered to one filing, and says so when a filing isn't in the collection
# (Lyft FY2020-2022 only). Without this, retrieval silently returns other years' chunks,
# and the LLM fills the gap with figures from memory.
print(search_10k("Lyft 2024 total revenue"))

There is no Lyft, Inc. 10-K for fiscal year 2024 in the collection. Available fiscal years: [2020, 2021, 2022].


## 2. The prompt

The whole "agent" is this text: the tool list, the exact **Thought / Action / Action Input** format, and the rules that keep it grounded.

In [4]:
SYSTEM_PROMPT = f"""You are a ReAct agent. Answer the question by reasoning step by step and using tools.

Tools:
{chr(10).join(f"- {name}: {desc}" for name, (_, desc) in TOOLS.items())}

On every turn, respond in EXACTLY one of these two formats and nothing else:

Thought: <your reasoning about what to do next>
Action: <one tool name from the list>
Action Input: <the input for the tool>

or, once you have enough information:

Thought: <your final reasoning>
Final Answer: <the answer to the original question>

Rules:
- Do ONE action per turn, then stop. The Observation will be given to you.
- Never write an Observation yourself.
- Break multi-part questions into several focused searches.
- Use the calculator for any arithmetic instead of doing it in your head.
- In calculations, use the exact figures from the financial statements (e.g. revenue 17,455 in $ millions),
  not rounded figures from the narrative text (e.g. "$17.5 billion").
- In the Final Answer, state every figure you used (with its source) as well as the result.
- Always use at least one tool before a Final Answer. Never answer from memory.
- Only state facts and figures that appear in an Observation. If the documents don't contain
  something, say so plainly (or use search_web when current or external data is appropriate).
- If an Observation doesn't contain what you need, rephrase and search again (e.g. use the
  filing's own wording) before concluding the information isn't there.
"""

print(SYSTEM_PROMPT)

You are a ReAct agent. Answer the question by reasoning step by step and using tools.

Tools:
- search_10k: Search the 10-K filings (Uber FY2021; Lyft FY2020-FY2022). Returns raw excerpts. Search for ONE fact per call with a SHORT query: ONE company, the fiscal year and a few key words, e.g. 'Lyft 2022 total revenue'.
- search_openai_docs: Search the OpenAI Agents SDK documentation.
- search_web: Search the live internet for current or external information.
- calculator: Evaluate an arithmetic expression, e.g. (17455 - 11139) / 11139 * 100

On every turn, respond in EXACTLY one of these two formats and nothing else:

Thought: <your reasoning about what to do next>
Action: <one tool name from the list>
Action Input: <the input for the tool>

or, once you have enough information:

Thought: <your final reasoning>
Final Answer: <the answer to the original question>

Rules:
- Do ONE action per turn, then stop. The Observation will be given to you.
- Never write an Observation yourself.
- Br

## 3. The loop

1. Send the conversation to the model.
2. Parse its reply: a **Final Answer** ends the loop; an **Action** runs the tool.
3. Append the tool result as an **Observation** and go again, up to `MAX_STEPS`.

Every step is printed, so you can watch the reasoning happen.

In [5]:
def _parse(text: str):
    """Return ('final', answer) or ('action', tool, tool_input) or ('error', msg)."""
    final = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
    if final:
        return ("final", final.group(1).strip())
    action = re.search(r"Action:\s*(.+)", text)
    action_input = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)
    if action and action_input:
        return ("action", action.group(1).strip(), action_input.group(1).strip())
    return ("error", "Could not parse. Use the Thought/Action/Action Input or Final Answer format.")


# A Final Answer saying the documents lack something, e.g. "the excerpt does not state the purchase price".
GAVE_UP = re.compile(r"(?i)(does|do|did) not (state|provide|contain|include|disclose)|not (available|provided|stated)"
                     r"|cannot be (calculated|determined)|unable to (determine|find|calculate)")


def react_agent(question: str, max_steps: int = MAX_STEPS, verbose: bool = True, on_step=None) -> str:
    """on_step(step, kind, text), if given, is called with kind 'reply' or 'observation' (used by app.py)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}"},
    ]
    if verbose:
        print(f"\n{'=' * 70}\n❓ {question}\n{'=' * 70}")

    used_tool, searches = False, 0
    for step in range(1, max_steps + 1):
        try:
            reply = rag_helpers._openaiclient.chat.completions.create(
                model=MODEL, messages=messages
            ).choices[0].message.content
        except Exception as e:  # e.g. rate limit or a rejected request: stop cleanly, keep the trace
            return f"Stopped: model API error at step {step}: {e}"

        # If the model runs ahead and writes its own Observation, cut it off.
        reply = reply.split("Observation:")[0].strip()
        messages.append({"role": "assistant", "content": reply})

        if verbose:
            print(f"\n--- Step {step} ---\n{reply}")
        if on_step:
            on_step(step, "reply", reply)

        parsed = _parse(reply)
        if parsed[0] == "final" and not used_tool:
            parsed = ("error", "You have not used any tool yet. Call a tool before giving a Final Answer.")
        elif parsed[0] == "final" and searches < 3 and GAVE_UP.search(parsed[1]):
            parsed = ("error", "Before concluding the documents don't contain it, search again with a SHORT, "
                               "differently-worded query that uses the filing's own wording.")
        if parsed[0] == "final":
            if verbose:
                print(f"\n✅ Done in {step} step(s).")
            return parsed[1]

        if parsed[0] == "error":
            observation = parsed[1]
        else:
            _, tool, tool_input = parsed
            if tool not in TOOLS:
                observation = f"Unknown tool '{tool}'. Choose from: {', '.join(TOOLS)}"
            else:
                used_tool = True
                searches += tool.startswith("search_")
                try:
                    observation = TOOLS[tool][0](tool_input)
                except Exception as e:
                    observation = f"Tool error: {e}"

        observation = observation[:13000]  # room for 6 raw chunks, but keep the context small
        if verbose:
            print(f"\nObservation: {observation}")
        if on_step:
            on_step(step, "observation", observation)
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "Stopped: reached max steps without a final answer."

## 4. The questions, and a side-by-side helper

Each question needs several lookups, more than one source, or a result that decides the next lookup. `compare()` sends a question to the plain agentic RAG (`_run_rag_pipeline`, without the cache) and to the ReAct agent, and prints the 3 chunks the one-shot retrieval saw.

In [6]:
# ── Questions a single-shot RAG call can't answer well, but a ReAct loop can ──────
# Each needs several retrievals, retrievals from different sources, and/or arithmetic.

HARD_QUESTIONS = [
    # Chained: find WHICH company was acquired, then its price, then revenue, then divide.
    # (One retrieval on the full question doesn't return the acquisition chunk.)
    "According to Uber's 2021 10-K, which freight logistics company did Uber acquire in 2021, how much did "
    "Uber pay for it, and what percentage of Uber's 2021 revenue was that price?",

    # Chained: the answer to part 1 (which company) decides what to look up in part 2.
    "Which company reported higher annual revenue in its 10-K: Uber (2021) or Lyft (2022)? For that company, "
    "find its research and development expense for the same year and express it as a percentage of its revenue.",

    # Two figures from different sections of the filing (debt note + income statement), then arithmetic.
    # Obscure enough that the model can't fill the gap from memory.
    "According to Lyft's 2022 10-K, what was Lyft's long-term debt, net of current portion, at "
    "December 31, 2022, and what percentage of Lyft's 2022 revenue is that?",

    # Mixes the 10-K collection with live web data. The router picks only ONE source.
    "Uber reported its 2021 revenue in its 10-K. Using Uber's current market cap from the web, "
    "what is its approximate price-to-sales ratio against that 2021 revenue?",

    # Mixes OpenAI docs with 10-K content: design question grounded in two sources.
    "Name two risk factors from Uber's 2021 10-K, then explain which OpenAI Agents SDK feature "
    "(e.g. guardrails, handoffs, tracing) would help an agent that answers questions about those risks.",

    # 10-K figure first, then a web lookup that builds on it, then arithmetic.
    "According to its FY2021 10-K, how many MAPCs did Uber have in Q4 2021? Then look up on the web how many "
    "MAPCs Uber reported for Q2 2026, and calculate how much that metric has grown since Q4 2021.",
]


import asyncio
import rag_helpers
from rag_helpers import _run_rag_pipeline, _get_text_embeddings


def show_retrieved_chunks(question: str, collection: str = "10k_data") -> None:
    """Print the top-3 chunks a single retrieval on the full question returns."""
    hits = asyncio.run(rag_helpers._qdrant.query_points(
        collection_name=collection, query=_get_text_embeddings(question), limit=3))
    for p in hits.points:
        meta = p.payload.get("metadata", {})
        preview = " ".join(p.payload["content"].split())[:160]
        print(f"   • [{meta.get('company')} {meta.get('fiscal_year')}] {preview}...")


def compare(question: str) -> None:
    """Same question → plain agentic RAG (one shot) vs ReAct agent (loop), then the golden-answer check."""
    print("🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer\n")
    print("   Chunks a one-shot 10-K retrieval sees for this question:")
    show_retrieved_chunks(question)
    print()
    plain_answer = _run_rag_pipeline(question)
    print(plain_answer)
    print("\n\n🔸 REACT AGENT — Thought → Action → Observation loop")
    answer = react_agent(question)
    print(f"\n🧾 FINAL: {answer}")

    gold = GOLDEN.get(question)
    if gold:
        print(f"\n📏 GOLDEN ANSWER ({gold['id']}): {gold['answer']}")
        scores = {}
        for who, ans in (("Plain RAG", plain_answer), ("ReAct", answer)):
            res = check_answer(ans, gold)
            scores[who] = (sum(passed for _, passed in res), len(res))
            print(f"   {who:9} {scores[who][0]}/{scores[who][1]}  " + "   ".join(
                ("✅ " if passed else "❌ ") + f["label"] + (" ⏱" if f.get("time_dependent") else "") for f, passed in res))
        SCORES[gold["id"]] = scores

for i, q in enumerate(HARD_QUESTIONS, 1):
    print(f"{i}. {q}")

1. According to Uber's 2021 10-K, which freight logistics company did Uber acquire in 2021, how much did Uber pay for it, and what percentage of Uber's 2021 revenue was that price?
2. Which company reported higher annual revenue in its 10-K: Uber (2021) or Lyft (2022)? For that company, find its research and development expense for the same year and express it as a percentage of its revenue.
3. According to Lyft's 2022 10-K, what was Lyft's long-term debt, net of current portion, at December 31, 2022, and what percentage of Lyft's 2022 revenue is that?
4. Uber reported its 2021 revenue in its 10-K. Using Uber's current market cap from the web, what is its approximate price-to-sales ratio against that 2021 revenue?
5. Name two risk factors from Uber's 2021 10-K, then explain which OpenAI Agents SDK feature (e.g. guardrails, handoffs, tracing) would help an agent that answers questions about those risks.
6. According to its FY2021 10-K, how many MAPCs did Uber have in Q4 2021? Then look 

## 5. Golden answers

`golden_answers.json` holds the expected answer to each question, its **key facts** (each checked with a regex, so `$2.3 billion`, `$2.3B` and `2,300` all count) and the **exact 10-K quotes** they come from. `compare()` checks both agents' answers against it.

⏱ marks facts that come from the live web (market cap, latest MAPCs). They were recorded on 2026-09-23 and can legitimately change.

In [7]:
import json
import re
from IPython.display import Markdown, display

GOLDEN = {g["question"]: g for g in json.load(open("golden_answers.json"))["questions"]}
SCORES = {}  # filled in by compare()


def check_answer(answer: str, gold: dict) -> list:
    """[(fact, passed)] for each key fact; a fact passes when >= fact['min'] of its regexes match."""
    text = answer.replace("**", "").replace("\\$", "$").replace("\\%", "%")
    return [(f, sum(bool(re.search(p, text)) for p in f["accept"]) >= f.get("min", 1)) for f in gold["key_facts"]]


esc = lambda s: s.replace("$", "\\$")  # keep Jupyter from reading $ amounts as LaTeX
rows = ["| # | Golden answer | Key facts checked | Evidence (quoted from the sources) |", "|---|---|---|---|"]
for g in GOLDEN.values():
    facts = "<br>".join(f["label"] + (" ⏱" if f.get("time_dependent") else "") for f in g["key_facts"])
    quotes = "<br>".join(f"*{s['filing']}:* “{s['quote']}”" for s in g["sources"])
    rows.append(f"| {g['id']} | {esc(g['answer'])} | {esc(facts)} | {esc(quotes)} |")
display(Markdown("\n".join(rows)))

| # | Golden answer | Key facts checked | Evidence (quoted from the sources) |
|---|---|---|---|
| Q1 | Transplace, for \$2.3 billion (acquisition-date fair value of consideration), which is about 13.2% of Uber's 2021 revenue of \$17,455 million. | Company: Transplace<br>Price: \$2.3 billion<br>Share of revenue: ≈13.1–13.2% | *Uber 10-K FY2021:* “we completed the acquisition of Transplace.”<br>*Uber 10-K FY2021:* “The acquisition date fair value of the consideration transferred for Transplace was \$ 2.3 billion.”<br>*Uber 10-K FY2021:* “Year Ended December 31, 2019 2020 2021 Revenue \$ 13,000 \$ 11,139 \$ 17,455” |
| Q2 | Uber: \$17,455 million (2021) vs Lyft \$4,095,135 thousand (2022). Uber's 2021 R&D was \$2,054 million, about 11.8% of revenue. | Uber 2021 revenue: \$17,455M<br>Lyft 2022 revenue: \$4,095M<br>Uber 2021 R&D: \$2,054M<br>R&D share of revenue: ≈11.8% | *Uber 10-K FY2021:* “Revenue \$ 13,000 \$ 11,139 \$ 17,455”<br>*Uber 10-K FY2021:* “Research and development 4,836 2,205 2,054”<br>*Lyft 10-K FY2022:* “Total revenue \$ 4,095,135 \$ 3,208,323 \$ 2,364,681” |
| Q3 | \$803,207 thousand (≈\$803.2 million), about 19.6% of Lyft's 2022 revenue of \$4,095,135 thousand. | Long-term debt, net of current portion: \$803.2M<br>Share of revenue: ≈19.6% | *Lyft 10-K FY2022:* “Long-term debt, net of current portion 803,207 655,173”<br>*Lyft 10-K FY2022:* “Less: long-term debt maturing within one year 36,287 56,264 Total long-term debt \$ 803,207 \$ 655,173”<br>*Lyft 10-K FY2022:* “Total revenue \$ 4,095,135 \$ 3,208,323 \$ 2,364,681” |
| Q4 | Market cap ÷ \$17,455 million. On 2026-09-23 the web reported a market cap of about \$142.6B, giving roughly 8.2×. | Uber 2021 revenue: \$17,455M<br>P/S ≈ 8.2× (market cap ≈ \$142.6B on 2026-09-23) ⏱ | *Uber 10-K FY2021:* “Revenue \$ 13,000 \$ 11,139 \$ 17,455”<br>*Web (companiesmarketcap.com, 2026-09-23):* “As of September 2026 Uber has a market cap of \$142.63 Billion USD.” |
| Q5 | Any two risk factors stated in the filing, e.g. driver classification as employees, data privacy/processing, COVID-19, new products and features, operational metrics, competition, regulation, security breaches; plus a relevant SDK feature (guardrails, tracing or handoffs) with a reason. | At least two risk factors that appear in the 10-K<br>Names a relevant Agents SDK feature | *Uber 10-K FY2021:* “Our business would be adversely affected if Drivers were classified as employees, workers or quasi-employees instead of independent contractors.”<br>*Uber 10-K FY2021:* “We face risks related to our collection, use, transfer, disclosure, and other processing of data”<br>*Uber 10-K FY2021:* “If we are unable to successfully introduce new or upgraded products, offerings, or features”<br>*Uber 10-K FY2021:* “Certain of our operational metrics are subject to inherent challenges in measurement, and real or perceived inaccuracies in such metrics may harm our reputation”<br>*OpenAI docs:* “Building guardrails: Set up guardrails that address the risks you've already identified for your use case” |
| Q6 | 118 million MAPCs in Q4 2021 (10-K) and 208 million in Q2 2026 (web): growth of about 76%. | Q4 2021 MAPCs: 118 million<br>Q2 2026 MAPCs: 208 million<br>Growth since Q4 2021: ≈76% | *Uber 10-K FY2021:* “Monthly Active Platform Consumers (“MAPCs”) (2), (3) 93 118 27 %”<br>*Web (investor.uber.com, 2026-09-23):* “Uber 208 million · monthly active platform consumers* (*For the qu…” |

## 6. Plain agentic RAG vs ReAct: the six questions

In [8]:
# Q1 — chained: which company → its price → % of revenue
compare(HARD_QUESTIONS[0])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:


   • [Uber Technologies, Inc. 2021] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 billion in 2021, up 56%, or 53% o...
   • [Uber Technologies, Inc. 2021] 2021 Revenue $ 11,139 $ 17,455 57 % 2021 Compared to 2020 Revenue increased $6.3 billion, or 57%, primarily attributable to an increase in Gross Bookings of 56%...
   • [Uber Technologies, Inc. 2021] In October 2020, Freight Holding entered into a Series A preferred stock purchase agreement (“2020 Freight Series A Preferred Stock Purchase Agreement”) with ou...



📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber's 2021 10-K acquisition and financial figures.


Uber acquired **Transplace**, a freight-logistics company, in 2021 for **$2.25 billion**. Uber’s 2021 revenue was **$17.455 billion** [1], so the acquisition price represented:

\[
\frac{\$2.25\text{ billion}}{\$17.455\text{ billion}} \times 100 \approx 12.9\%
\]

Thus, Transplace cost about **12.9% of Uber’s 2021 revenue**. [1][2]


🔸 REACT AGENT — Thought → Action → Observation loop

❓ According to Uber's 2021 10-K, which freight logistics company did Uber acquire in 2021, how much did Uber pay for it, and what percentage of Uber's 2021 revenue was that price?



--- Step 1 ---
Thought: I need to search Uber's 2021 10-K for the freight logistics acquisition and purchase price, then find total 2021 revenue and calculate the price as a percentage of revenue. I will start with the acquisition details.
Action: search_10k
Action Input: Uber 2021 freight logistics company acquisition purchase price



Observation: [1] (Uber Technologies, Inc. FY2021) In October 2020, Freight Holding entered into a Series A preferred stock purchase agreement (“2020 Freight Series A Preferred Stock Purchase Agreement”) with outside investor (“2020 Freight Series A Investor”) to sell shares of Series A Preferred Stock (“Freight Series A”). In July 2021, we entered into a Freight Series A preferred stock purchase agreement and sold shares of Freight Series A to The Public Investment Fund, which is an investor in Uber. In November 2021, Freight Holding entered into a series A-1 stock purchase agreement (“2021 Series A-1 Preferred Stock Purchase Agreement”) with outside investors (“Freight Series A-1 Investors”) to sell shares of Series A-1 convertible preferred stock of Freight Holding (“Freight Series A-1”). Neither the Freight Series A or Freight Series A-1 investments changed the conclusion that Freight Holding is a consolidated VIE. As of December 31, 2020 and 2021, we continue to own the majority o


--- Step 2 ---
Thought: The filing identifies Transplace as the acquired freight logistics company, but the excerpt does not include its purchase price. I need to search specifically for Transplace’s consideration transferred or total purchase price.
Action: search_10k
Action Input: Uber 2021 Transplace total purchase price consideration transferred

Observation: [1] (Uber Technologies, Inc. FY2021) through Transplace’s expertise in transportation management. The acquisition of Transplace has been accounted for as a business combination. The acquisition date fair value of the consideration transferred for Transplace was $ 2.3 billion. The followin g table summarizes the preliminary fair value of assets acquired and liabilities assumed as of the date of acquisition (in millions) : Fair Value Cash and cash equivalents $ 29 Accounts receivable, net 899 Prepaid expenses and other current assets 23 Property and equipment, net 44 Operating lease right-of-use assets 57 Intangible assets, net


--- Step 3 ---
Thought: I have the acquisition and price: Transplace, with consideration transferred of $2.3 billion. I now need Uber’s exact 2021 total revenue from the 10-K to calculate the percentage.
Action: search_10k
Action Input: Uber 2021 total revenue 2021 consolidated statements operations exact revenue million

Observation: [1] (Uber Technologies, Inc. FY2021) Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown separately below 6,061 5,154 9,351 Operations and support 2,302 1,819 1,877 Sales and marketing 4,626 3,583 4,789 Research and development 4,836 2,205 2,054 General and administrative 3,299 2,666 2,316 Depreciation and amortization 472 575 902 Total costs and expenses 21,596 16,002 21,289 Loss from operations ( 8,596 ) ( 4,863 ) ( 3,834 ) Interest expense ( 559 ) ( 458 ) ( 483 ) Other income (expense), net 722 ( 1,625 ) 3,292 Loss before income taxes and loss fr


--- Step 4 ---
Thought: The 10-K reports Transplace as the acquired freight logistics company, with acquisition-date consideration of $2.3 billion, and 2021 revenue of $17,455 million. I will calculate $2,300 million divided by $17,455 million, multiplied by 100.
Action: calculator
Action Input: 2300 / 17455 * 100

Observation: 13.176740189057575



--- Step 5 ---
Thought: The calculation is 2,300 ÷ 17,455 × 100 = 13.1767%, so the acquisition price represented approximately 13.2% of 2021 revenue.
Final Answer: Uber acquired **Transplace**, a transportation management and third-party logistics provider, in 2021. According to Uber’s FY2021 10-K, the **acquisition-date consideration transferred was $2.3 billion** (approximately **$2,300 million**) in an all-cash transaction. Uber’s **2021 revenue was $17,455 million**. Therefore:

**$2,300 million ÷ $17,455 million × 100 = 13.1767%**

Thus, Uber paid approximately **13.2% of its 2021 revenue** for Transplace.

✅ Done in 5 step(s).

🧾 FINAL: Uber acquired **Transplace**, a transportation management and third-party logistics provider, in 2021. According to Uber’s FY2021 10-K, the **acquisition-date consideration transferred was $2.3 billion** (approximately **$2,300 million**) in an all-cash transaction. Uber’s **2021 revenue was $17,455 million**. Therefore:

**$2,300 million ÷ $17,4

In [9]:
# Q2 — chained: which company → its R&D → % of revenue
compare(HARD_QUESTIONS[1])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:
   • [Lyft, Inc. 2020] network of Light Vehicles, and Lyft Rentals, which is recognized in accordance with Accounting Standards Codification Topic 842 (“ASC 842”). The table below pre...
   • [Uber Technologies, Inc. 2021] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 billion in 2021, up 56%, or 53% o...
   • [Lyft, Inc. 2021] their first ride on the Lyft Platform. The Company records the incentive as a liability at the time the incentive is earned by the referrer with the correspondi...



📍 Route: 10K_DOCUMENT_QUERY  |  Compares Uber and Lyft 10-K financials and R&D expense.


Uber reported the higher annual revenue:

- **Uber (2021):** $17.455 billion in revenue [1]  
- **Lyft (2022):** $4.095 billion in revenue [2]

For Uber in 2021, research and development expense was **$2.070 billion** [1].

\[
\text{R\&D as \% of revenue}=\frac{2.070}{17.455}\times100\approx \mathbf{11.9\%}
\]

**Answer: Uber; its 2021 R&D expense was approximately 11.9% of revenue.**

[1] Uber Technologies, Inc., 2021 Form 10-K, Consolidated Statements of Operations.  
[2] Lyft, Inc., 2022 Form 10-K, Consolidated Statements of Operations.


🔸 REACT AGENT — Thought → Action → Observation loop

❓ Which company reported higher annual revenue in its 10-K: Uber (2021) or Lyft (2022)? For that company, find its research and development expense for the same year and express it as a percentage of its revenue.



--- Step 1 ---
Thought: I need to find Uber’s fiscal 2021 total revenue from its 10-K.  
Action: search_10k  
Action Input: Uber 2021 total revenue



Observation: [1] (Uber Technologies, Inc. FY2021) Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown separately below 6,061 5,154 9,351 Operations and support 2,302 1,819 1,877 Sales and marketing 4,626 3,583 4,789 Research and development 4,836 2,205 2,054 General and administrative 3,299 2,666 2,316 Depreciation and amortization 472 575 902 Total costs and expenses 21,596 16,002 21,289 Loss from operations ( 8,596 ) ( 4,863 ) ( 3,834 ) Interest expense ( 559 ) ( 458 ) ( 483 ) Other income (expense), net 722 ( 1,625 ) 3,292 Loss before income taxes and loss from equity method investments ( 8,433 ) ( 6,946 ) ( 1,025 ) Provision for (benefit from) income taxes 45 ( 192 ) ( 492 ) Loss from equity method investments ( 34 ) ( 34 ) ( 37 ) Net loss including non-controlling interests ( 8,512 ) ( 6,788 ) ( 570 ) Less: net loss attributable to non-controlling interests, net of tax ( 6 )


--- Step 2 ---
Thought: I have Uber’s 2021 revenue ($17,455 million) and R&D expense ($2,054 million). I need Lyft’s 2022 revenue to determine which company reported more.  
Action: search_10k  
Action Input: Lyft 2022 total revenue

Observation: [1] (Lyft, Inc. FY2022) 250,344 156,025 Total revenue $ 4,095,135 $ 3,208,323 $ 2,364,681 84 Revenue from Contracts with Customers (ASC 606) The Company recognizes revenue for its rideshare marketplace in accordance with ASC 606. The Company generates revenue from service fees and commissions (collectively, “fees”) paid by drivers for use of the Lyft Platform and related activities to connect drivers with riders to facilitate and successfully complete rides via the Lyft App where the Company operates as a TNC. The Company recognizes revenue upon completion of each ride. Drivers enter into terms of service (“ToS”) with the Company in order to use the Lyft Driver App. Under the ToS, drivers agree that the Company retains the applicable fee as c


--- Step 3 ---
Thought: Uber reported higher 2021 revenue of $17,455 million versus Lyft’s 2022 revenue of $4,095.135 million. I need to calculate Uber’s 2021 R&D expense as a percentage of revenue.  
Action: calculator  
Action Input: 2054 / 17455 * 100

Observation: 11.767401890575767



--- Step 4 ---
Thought: Uber reported higher annual revenue. Its 2021 revenue was $17,455 million, compared with Lyft’s 2022 revenue of $4,095.135 million. Uber’s 2021 research and development expense was $2,054 million; $2,054 ÷ $17,455 × 100 = 11.7674%, or approximately 11.77%.  
Final Answer: **Uber** reported higher annual revenue: **$17,455 million in 2021**, versus Lyft’s **$4,095.135 million in 2022**. Uber’s 2021 research and development expense was **$2,054 million**, which was **approximately 11.77% of revenue**. (Source: Uber FY2021 and Lyft FY2022 10-K consolidated statements of operations.)

✅ Done in 4 step(s).

🧾 FINAL: **Uber** reported higher annual revenue: **$17,455 million in 2021**, versus Lyft’s **$4,095.135 million in 2022**. Uber’s 2021 research and development expense was **$2,054 million**, which was **approximately 11.77% of revenue**. (Source: Uber FY2021 and Lyft FY2022 10-K consolidated statements of operations.)

📏 GOLDEN ANSWER (Q2): Uber: $17,455 milli

In [10]:
# Q3 — two obscure figures from different parts of the filing
compare(HARD_QUESTIONS[2])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:
   • [Lyft, Inc. 2022] Year Ended December 31, 2022 2021 2020 Reconciliation of cash, cash equivalents and restricted cash and cash equivalents to the consolidated balance sheets Cash...
   • [Lyft, Inc. 2021] Year Ended December 31, 2021 2020 2019 Reconciliation of cash, cash equivalents and restricted cash and cash equivalents to the consolidated balance sheets Cash...
   • [Lyft, Inc. 2021] Operations and support 24,083 15,829 75,212 Research and development 414,324 325,624 971,941 Sales and marketing 38,243 23,385 72,046 General and administrative...



📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Lyft's 2022 10-K financial figures and revenue percentage.


Lyft reported **long-term debt, net of current portion, of approximately $1.03 billion** at December 31, 2022. Its 2022 revenue was approximately **$4.095 billion**.  

Calculation:

\[
\frac{\$1.03\text{ billion}}{\$4.095\text{ billion}} \times 100 \approx 25.1\%
\]

Thus, Lyft’s long-term debt represented **approximately 25% of its 2022 revenue**.[1][2]


🔸 REACT AGENT — Thought → Action → Observation loop

❓ According to Lyft's 2022 10-K, what was Lyft's long-term debt, net of current portion, at December 31, 2022, and what percentage of Lyft's 2022 revenue is that?



--- Step 1 ---
Thought: I need to find Lyft’s long-term debt, net of current portion, at December 31, 2022, and Lyft’s exact 2022 revenue in the 2022 10-K. I’ll search for the debt figure first.
Action: search_10k
Action Input: Lyft 2022 long-term debt net of current portion December 31 2022



Observation: [1] (Lyft, Inc. FY2022) Year Ended December 31, 2022 2021 2020 Reconciliation of cash, cash equivalents and restricted cash and cash equivalents to the consolidated balance sheets Cash and cash equivalents $ 281,090 $ 457,325 $ 319,734 Restricted cash and cash equivalents 109,368 73,205 118,559 Restricted cash, included in prepaid expenses and other current assets 1,364 663 192 Total cash, cash equivalents and restricted cash and cash equivalents $ 391,822 $ 531,193 $ 438,485 Supplemental disclosures of cash flow information Cash paid for income taxes $ 10,723 $ 5,865 $ 4,037 Cash paid for interest 16,752 16,521 12,545 Non-cash investing and financing activities Financed vehicles acquired, net of principal payments $ 48,104 $ 56,830 $ 34,051 Purchases of property and equipment, and scooter fleet not yet settled 31,534 12,214 7,220 Contingent consideration 15,000 — — Right-of-use assets acquired under finance leases 11,428 26,640 6,556 Right-of-use assets acquired under op


--- Step 2 ---
Thought: The 2022 10-K reports long-term debt, net of current portion of $803,207 thousand and 2022 revenue of $4,095,135 thousand. I’ll calculate debt as a percentage of revenue.
Action: calculator
Action Input: 803207 / 4095135 * 100

Observation: 19.613687949237328



--- Step 3 ---
Thought: The 10-K balance sheet reports long-term debt, net of current portion of $803,207 thousand at December 31, 2022, and the consolidated statements of operations report 2022 revenue of $4,095,135 thousand. The calculator gives 19.613687949237328%, which rounds to 19.6%.
Final Answer: Lyft’s long-term debt, net of current portion, was **$803.207 million** at December 31, 2022 (Lyft 2022 10-K, consolidated balance sheets). Lyft’s 2022 revenue was **$4,095.135 million** (same 10-K, consolidated statements of operations).

Calculation: **$803.207 million ÷ $4,095.135 million × 100 = 19.6137%**, or approximately **19.6% of 2022 revenue**.

✅ Done in 3 step(s).

🧾 FINAL: Lyft’s long-term debt, net of current portion, was **$803.207 million** at December 31, 2022 (Lyft 2022 10-K, consolidated balance sheets). Lyft’s 2022 revenue was **$4,095.135 million** (same 10-K, consolidated statements of operations).

Calculation: **$803.207 million ÷ $4,095.135 million × 100 = 19.

In [11]:
# Q4 — 10-K + live web (two sources)
compare(HARD_QUESTIONS[3])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:
   • [Uber Technologies, Inc. 2021] by analogy. The standard is effective for public companies for fiscal years beginning after December 15, 2021. Early adoption is permitted. We are currently eva...
   • [Uber Technologies, Inc. 2021] Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown s...
   • [Uber Technologies, Inc. 2021] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 billion in 2021, up 56%, or 53% o...



📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber’s 2021 10-K revenue and price-to-sales ratio.


Uber’s 2021 revenue was **$17.455 billion**.[1]

Using a current market capitalization of approximately **$170 billion**,[2]:

\[
\text{Price-to-sales ratio}=\frac{\$170\text{B}}{\$17.455\text{B}}\approx 9.7\times
\]

**Approximate price-to-sales ratio: 9.7× (roughly 10×).**

[1] [Uber 2021 Form 10-K — Consolidated Statements of Operations](https://www.sec.gov/Archives/edgar/data/1543151/000154315122000014/uber-20211231.htm)  
[2] [Uber market capitalization](https://companiesmarketcap.com/uber/marketcap/)


🔸 REACT AGENT — Thought → Action → Observation loop

❓ Uber reported its 2021 revenue in its 10-K. Using Uber's current market cap from the web, what is its approximate price-to-sales ratio against that 2021 revenue?



--- Step 1 ---
Thought: I need Uber’s exact 2021 revenue from its 10-K first. 
Action: search_10k
Action Input: Uber FY2021 total revenue 2021 annual report

Observation: [1] (Uber Technologies, Inc. FY2021) Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown separately below 6,061 5,154 9,351 Operations and support 2,302 1,819 1,877 Sales and marketing 4,626 3,583 4,789 Research and development 4,836 2,205 2,054 General and administrative 3,299 2,666 2,316 Depreciation and amortization 472 575 902 Total costs and expenses 21,596 16,002 21,289 Loss from operations ( 8,596 ) ( 4,863 ) ( 3,834 ) Interest expense ( 559 ) ( 458 ) ( 483 ) Other income (expense), net 722 ( 1,625 ) 3,292 Loss before income taxes and loss from equity method investments ( 8,433 ) ( 6,946 ) ( 1,025 ) Provision for (benefit from) income taxes 45 ( 192 ) ( 492 ) Loss from equity method investments ( 34 ) ( 3


--- Step 2 ---
Thought: Uber’s FY2021 10-K reports revenue of $17,455 million. I now need Uber’s current market capitalization from a live web source. 
Action: search_web
Action Input: Uber current market capETwitter
Getting your response from the internet 🌐 ...



Observation: [1] Uber sits at $139B market cap, while Waymo holds a $126 ...
    Uber sits at $139B market cap, while Waymo holds a $126B valuation. Uber generates $54B in annual revenue, compared to Waymo's $0.4B ($400M).
    Source: https://www.instagram.com/p/Ddam430xAnc/

[2] AUR vs TWTR - Aurora Innovation Inc. vs Twitter Inc. side- ...
    Price $6.32+7.94% $53.83+0.77% Exchange. Uber Technologies, Inc sold $471,600,000 worth of shares (72,000,000 units at $6.55)
    Source: https://www.quantisnow.com/compare/AUR/vs/TWTR

[3] Uber Technologies (LON:0A1U) Stock Price & Overview
    Get the latest Uber Technologies, Inc. (LON:0A1U) stock price quote ... Market Cap, 114.61B -20.5%. Revenue (ttm), 41.62B +16.7%. Net Income, 7.22B -24.1 ...
    Source: https://stockanalysis.com/quote/lon/0A1U/

[4] Uber (UBER) Stock Price Today & Analysis
    Market cap $142.75B Price / Earnings ratio 14.9x Price / Sales ratio 2.6x Price / Book ratio 5.2x Return on assets 0.1%
    Source: https://www


--- Step 3 ---
Thought: Using the web-reported market cap of approximately $139 billion and the FY2021 10-K revenue of $17,455 million ($17.455 billion), I will calculate market cap divided by revenue. 
Action: calculator
Action Input: 139 / 17.455

Observation: 7.9633342881695794



--- Step 4 ---
Thought: The calculation is $139 billion ÷ $17.455 billion = 7.9633, so the approximate price-to-sales ratio is 8.0x.
Final Answer: Uber’s approximate price-to-sales ratio is **8.0x**:

- **Current market cap:** approximately **$139 billion** (web source: Instagram result)
- **2021 revenue:** **$17.455 billion** (Uber FY2021 10-K; reported as $17,455 million)
- **Calculation:** $139 billion ÷ $17.455 billion = **7.96x**, or approximately **8.0x**

✅ Done in 4 step(s).

🧾 FINAL: Uber’s approximate price-to-sales ratio is **8.0x**:

- **Current market cap:** approximately **$139 billion** (web source: Instagram result)
- **2021 revenue:** **$17.455 billion** (Uber FY2021 10-K; reported as $17,455 million)
- **Calculation:** $139 billion ÷ $17.455 billion = **7.96x**, or approximately **8.0x**

📏 GOLDEN ANSWER (Q4): Market cap ÷ $17,455 million. On 2026-09-23 the web reported a market cap of about $142.6B, giving roughly 8.2×.
   Plain RAG 1/2  ✅ Uber 2021 revenue: $17,455

In [12]:
# Q5 — 10-K + OpenAI docs (two sources)
compare(HARD_QUESTIONS[4])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:
   • [Uber Technologies, Inc. 2021] proprietary technology applications supporting a variety of offerings on its platform (“platform(s)” or “Platform(s)”). Uber connects consumers (“Rider(s)”) wit...
   • [Uber Technologies, Inc. 2021] when using the Uber platform. Developing and delivering these new or upgraded products, offerings, and features is costly, and the success of such new products,...
   • [Uber Technologies, Inc. 2021] condition and cash flows, see the risk factor titled “-Our business would be adversely affected if Drivers were classified as employees, workers or quasi-employ...



📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber's 2021 10-K risk factors and an OpenAI Agents SDK feature.


Two risk factors in Uber’s 2021 Form 10-K are:

1. **Product and feature development risk:** Developing new or upgraded products is costly, and those products may not be completed on time, work as intended, provide value, or gain market acceptance. Uber notes that some California product changes reduced ride demand and Driver supply. [1]

2. **Operational-metrics risk:** Uber relies on internal systems and information from minority-owned affiliates to track metrics such as MAPCs, Trips, Gross Bookings, and category position. These metrics are not independently verified and may contain measurement inaccuracies that could damage Uber’s reputation and business. [1]

The most useful OpenAI Agents SDK feature would be **guardrails**. An agent answering questions about these risks could use guardrails to:

- Restrict answers to information grounded in the 10-K.
- Require a citation for each stated risk.
- Distinguish Uber’s disclosed risk from the agent’s interpretation.
- Flag uncertainty r


--- Step 1 ---
Thought: I need to find two specific risk factors in Uber’s 2021 10-K, beginning with one documented risk factor.
Action: search_10k
Action Input: Uber 2021 10-K risk factors cybersecurity risks

Observation: [1] (Uber Technologies, Inc. FY2021) 2020-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2020-01-01 2020-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2020-12-31 0001543151 us-gaap:AllowanceForCreditLossMember 2021-01-01 2021-12-31 0001543151 us-gaap:AllowanceForCreditLossMember 2021-12-31 0001543151 us-gaap:ValuationAllowanceOfDeferredTaxAssetsMember 2021-01-01 2021-12-31 0001543151 us-gaap:ValuationAllowanceOfDeferredTaxAssetsMember 2021-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2021-01-01 2021-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2021-12-31 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________ FORM 10-K ______________________________


--- Step 2 ---
Thought: I need the OpenAI Agents SDK documentation to connect these risks to an appropriate feature, so I will search for guardrails, handoffs, and tracing capabilities.
Action: search_openai_docs
Action Input: Agents SDK guardrails handoffs tracing

Observation: [1] (OpenAI docs) Building guardrails Set up guardrails that address the risks you’ve already identified for your use case and layer in additional ones as you uncover new vulnerabilities. We’ve found the following heuristic to be effective: 01 Focus on data privacy and content safety 02 Add new guardrails based on real-world edge cases and failures you encounter 03 Optimize for both security and user experience, tweaking your guardrails as your agent evolves. 27 A practical guide to building agents For example, here’s how you would set up guardrails when using the Agents SDK: Python 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 from import from import class str

[2] (OpenAI docs) customer_s


--- Step 3 ---
Thought: I have one clearly supported risk about driver classification and a second about data-security/privacy risks, but I should retrieve the exact risk-factor wording for the latter before answering.
Action: search_10k
Action Input: Uber 2021 10-K data security breach privacy risk factor

Observation: [1] (Uber Technologies, Inc. FY2021) 2020-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2020-01-01 2020-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2020-12-31 0001543151 us-gaap:AllowanceForCreditLossMember 2021-01-01 2021-12-31 0001543151 us-gaap:AllowanceForCreditLossMember 2021-12-31 0001543151 us-gaap:ValuationAllowanceOfDeferredTaxAssetsMember 2021-01-01 2021-12-31 0001543151 us-gaap:ValuationAllowanceOfDeferredTaxAssetsMember 2021-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2021-01-01 2021-12-31 0001543151 uber:SECSchedule1209ReserveInsuranceMember 2021-12-31 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington


--- Step 4 ---
Thought: The evidence supports two Uber FY2021 10-K risks—driver classification and data security/privacy—and the OpenAI documentation specifically supports guardrails for privacy, relevance, safety, and policy enforcement. I can now answer with the risks, sources, and feature rationale.

Final Answer: Two risk factors from Uber’s 2021 10-K are:

1. **Driver classification risk:** Uber’s business could be adversely affected if Drivers were classified as employees, workers, or quasi-employees. The filing says this could affect Uber’s financial condition, cash flows, revenue, cost of revenue, incentives, and promotions. *(Source: Uber Technologies, Inc. FY2021 Form 10-K, Item 1A, “Risk Factors”; excerpt identifying “Our business would be adversely affected if Drivers were classified as employees, workers or quasi-employees.”)*

2. **Data security and privacy risk:** Risks relating to Uber’s collection, use, transfer, disclosure, and other processing of data could lead to 

In [13]:
# Q6 — 10-K figure → web lookup that builds on it → growth
compare(HARD_QUESTIONS[5])

🔹 PLAIN AGENTIC RAG — one route, one retrieval, one answer

   Chunks a one-shot 10-K retrieval sees for this question:
   • [Uber Technologies, Inc. 2021] business, the limitations of these measures, and reconciliations of these measures to the most directly comparable GAAP financial measures, see the section titl...
   • [Uber Technologies, Inc. 2021] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 billion in 2021, up 56%, or 53% o...
   • [Uber Technologies, Inc. 2021] when using the Uber platform. Developing and delivering these new or upgraded products, offerings, and features is costly, and the success of such new products,...



📍 Route: 10K_DOCUMENT_QUERY  |  Asks about Uber 10-K financial metrics and growth.


According to Uber’s FY2021 Form 10-K, Uber had **118 million Monthly Active Platform Consumers (MAPCs) in Q4 2021**.[1]

Uber reported **211 million MAPCs in Q2 2026**.[2]

Calculation:

\[
211\text{ million} - 118\text{ million} = 93\text{ million}
\]

\[
\frac{93}{118}\times 100 \approx 78.8\%
\]

Thus, Uber’s MAPCs increased by **93 million consumers**, or approximately **79%**, from Q4 2021 to Q2 2026.

**Sources**

[1] Uber Technologies, Inc., *2021 Form 10-K*, quarterly operating metrics: [SEC filing](https://www.sec.gov/Archives/edgar/data/1543151/000154315122000014/uber-20211231.htm)  
[2] Uber Technologies, Inc., *Second Quarter 2026 Results*, investor-relations earnings release: [Uber Investor Relations](https://investor.uber.com/news-events/news/uber-announces-results-for-second-quarter-2026/)


🔸 REACT AGENT — Thought → Action → Observation loop

❓ According to its FY2021 10-K, how many MAPCs did Uber have in Q4 2021? Then look up on the web how many MAPCs Uber reported for


--- Step 1 ---
Thought: I’ll search Uber’s FY2021 10-K for its Q4 2021 MAPCs figure. 
Final Answer: Unable to complete the search.

Observation: You have not used any tool yet. Call a tool before giving a Final Answer.



--- Step 2 ---
Thought: I’ll search Uber’s FY2021 10-K for the Q4 2021 MAPCs figure. 
Action: search_10k
Action Input: Uber FY2021 Q4 2021 MAPCs mobile monthly active platform consumers


...


--- Step 3 ---
Thought: The excerpt defines MAPCs but does not show the Q4 2021 MAPCs value, so I’ll search the FY2021 filing for the quarterly MAPCs table. 
Action: search_10k
Action Input: Uber FY2021 MAPCs Q4 2021 118 million

Observation: [1] (Uber Technologies, Inc. FY2021) Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown separately below 6,061 5,154 9,351 Operations and support 2,302 1,819 1,877 Sales and marketing 4,626 3,583 4,789 Research and development 4,836 2,205 2,054 General and administrative 3,299 2,666 2,316 Depreciation and amortization 472 575 902 Total costs and expenses 21,596 16,002 21,289 Loss from operations ( 8,596 ) ( 4,863 ) ( 3,834 ) Interest expense ( 559 ) ( 458 ) ( 483 ) Other income (expense), net 722 ( 1,625 ) 3,292 Loss before income taxes and loss from equity method investments ( 8,433 ) ( 6,946 ) ( 1,025 ) Provision for (benefit from) income


--- Step 4 ---
Thought: The FY2021 10-K states that annual MAPCs are Q4 MAPCs and reports 118 million for 2021. I’ll now search the web for Uber’s Q2 2026 MAPCs. 
Action: search_web
Action Input: Uber Q2 2026 MAPCs reported mobile active platform consumers second quarter 2026
Getting your response from the internet 🌐 ...



Observation: [1] uber-20260630
    Highlights for the Second Quarter 2026 Trips increased to 3.9 billion in the second quarter of 2026, up 18% year-over-year, driven by MAPCs growth of 16% YoY ...
    Source: https://www.sec.gov/Archives/edgar/data/1543151/000154315126000032/uber-20260630.htm

[2] Uber Technologies (NYSE: UBER) Reports Strong Q2 Earnings With ...
    Uber Technologies, Inc. reported a GAAP earnings per share of $1.17 for the second quarter, topping analyst expectations by $0.34. ... Active Platform Consumers ( ...
    Source: https://kalkine.com/news/technology/uber-technologies-nyse-uber-reports-strong-q2-earnings-with-117-eps-above-estimates?srsltid=AU7gw4VWB3h_PHpyO43050PB2qiRWDurrtQbONf5OW_4jQfCp9HsoVKb

[3] Uber: Still A Generational Buying Opportunity
    Monthly active platform consumers (MAPCs) rose 16% YoY to 208 million in Q2, with those users taking nearly 3.9 billion rides over that time ...
    Source: https://seekingalpha.com/article/4934297-uber-still-a


--- Step 5 ---
Thought: Q4 2021 MAPCs were 118 million, and Q2 2026 MAPCs were reported as 208 million. I’ll calculate the absolute and percentage growth. 
Action: calculator
Action Input: (208 - 118) / 118 * 100

Observation: 76.27118644067797



--- Step 6 ---
Thought: The metric increased from 118 million to 208 million, an absolute increase of 90 million; the calculator gives percentage growth of 76.2712%.
Final Answer: Uber had **118 million MAPCs in Q4 2021**, according to its FY2021 10-K (the filing states that annual MAPCs represent Q4 MAPCs). Uber reported **208 million MAPCs in Q2 2026**, according to web reporting citing its Q2 results.

- Absolute growth: **208 million − 118 million = 90 million MAPCs**
- Percentage growth: **(208 − 118) / 118 × 100 = 76.27%**

Thus, MAPCs grew by **90 million**, or approximately **76.3%**, from Q4 2021 to Q2 2026.

✅ Done in 6 step(s).

🧾 FINAL: Uber had **118 million MAPCs in Q4 2021**, according to its FY2021 10-K (the filing states that annual MAPCs represent Q4 MAPCs). Uber reported **208 million MAPCs in Q2 2026**, according to web reporting citing its Q2 results.

- Absolute growth: **208 million − 118 million = 90 million MAPCs**
- Percentage growth: **(208 − 118) / 118 × 10

### Scoreboard: key facts matched, per question

In [14]:
rows = ["| # | Plain agentic RAG | ReAct |", "|---|---|---|"]
total = {"Plain RAG": [0, 0], "ReAct": [0, 0]}
for qid, s in SCORES.items():
    rows.append(f"| {qid} | {s['Plain RAG'][0]}/{s['Plain RAG'][1]} | {s['ReAct'][0]}/{s['ReAct'][1]} |")
    for who in total:
        total[who][0] += s[who][0]; total[who][1] += s[who][1]
rows.append(f"| **Total** | **{total['Plain RAG'][0]}/{total['Plain RAG'][1]}** | **{total['ReAct'][0]}/{total['ReAct'][1]}** |")
display(Markdown("\n".join(rows)))

| # | Plain agentic RAG | ReAct |
|---|---|---|
| Q1 | 1/3 | 3/3 |
| Q2 | 2/4 | 4/4 |
| Q3 | 0/2 | 2/2 |
| Q4 | 1/2 | 1/2 |
| Q5 | 2/2 | 2/2 |
| Q6 | 1/3 | 3/3 |
| **Total** | **7/16** | **15/16** |

## 7. What the comparison shows

Every figure in the ReAct answers below can be found in an Observation: a raw 10-K chunk, a docs chunk, or a web snippet. For plain RAG, the printed chunks show what it actually had to work with:

| # | Plain agentic RAG | ReAct agent |
|---|---|---|
| **1** | Its chunks contain **no purchase price and no revenue figure**, yet it answered *"Transplace, $2.25B, 12.9%"*. That's from memory, and the 10-K says **$2.3B**. | 3 searches find Transplace and *"consideration transferred … $2.3 billion"* in the filing, then revenue, then calculator: **13.2%**. |
| **2** | Its chunks contain **neither company's revenue nor Uber's R&D**, yet it gave all three figures, with R&D at **$2.070B** (the 10-K says $2,054M). Earlier runs gave $2.071B and $2.075B, so the figure changes from run to run because it comes from memory. | Uber revenue → Lyft FY2022 revenue ($4,095M) → decides it's Uber → the R&D line is in the retrieved income statement → calculator: **11.77%**. |
| **3** | Its chunks are about Lyft's **2021 expenses**; they contain **neither** figure, yet it answered *"$1.03B, 25.1% of revenue"*. In earlier runs it gave $1.73B, $1.99B, $2.19B and $2.73B: a different invented number each time. | One search finds the balance sheet (*"Long-term debt, net of current portion 803,207"*) and revenue ($4,095,135K), then calculator: **$803.2M ≈ 19.6%**. Correct in all 4 runs. |
| **4** | Routed to the 10-K only, yet stated a *"current market capitalization of approximately $170 billion"* with a citation. It never searched the web. | 10-K → web → calculator: **$139B ÷ $17.455B ≈ 8.0×**. The market cap is live, so it misses the golden 8.2× (recorded 2026-09-23), and this run's figure came from a social-media snippet: the web tool returns whatever ranks, and the agent doesn't judge the source. |
| **5** | Routed to the 10-K only; the Agents SDK half is from the model's memory. | Searched the 10-K twice (cybersecurity and data-privacy risk factors) and the docs once. Both score 2/2 here: this rubric only checks that the risks and the SDK feature are named, so it can't tell grounded from recalled. |
| **6** | Routed to the 10-K only, yet reported *"211 million MAPCs in Q2 2026"* with a citation. No web search happened, and the golden figure is 208M. | 10-K (118M, Q4 2021) → web (208M, Q2 2026) → calculator: **+76%**. |

**Takeaways**
- **One-shot RAG rarely says "I don't know".** When retrieval misses, the model fills the gap from memory or invents figures, and still cites the documents. Its answers sound right; only the retrieved chunks reveal they aren't grounded.
- **ReAct fixes this structurally:** one focused search per fact, later steps that depend on earlier results (Q1, Q2, Q6), several sources (Q4–Q6), and a calculator for the maths.
- **Grounding needs more than a loop.** Two design choices in the tool cell (section 1) matter as much as the loop itself:
  - Tools return **raw chunks**, not an LLM summary. With summaries, the agent's Observation for Q1 said "$2.25B" too.
  - `search_10k` **filters to one filing** and says when a filing isn't there (see the `Lyft 2024` cell in section 1). The collection has no Lyft 2023/2024 filing, and without the filter earlier runs quoted "Lyft 2024" figures from memory.

**Limitations**
- The collection only has what it has: Uber FY2021 and Lyft FY2020–2022, with no driver or rider counts for Lyft. Questions outside that must go to the web, or get an honest "not in the documents".
- The model occasionally adds stray tokens to an `Action Input` (visible in the traces). Searches still work because embedding search ignores the noise, but it's worth knowing when reading traces. A request can also be rejected by the API; the loop then stops cleanly with the error.
- Runs are non-deterministic; re-running will change some wording and occasionally the search path.